In [1]:
import sqlite3

from opentelemetry.sdk.trace.export import (
    SpanExporter,
    SpanExportResult,
)

class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)

        self.conn.execute("""
        CREATE TABLE IF NOT EXISTS spans(
            name TEXT,
            start_time INTEGER,
            end_time INTEGER,
            input_tokens INTEGER,
            output_tokens INTEGER,
            cost REAL
        )
        """)

        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})

            self.conn.execute(
                """
                INSERT INTO spans
                VALUES (?, ?, ?, ?, ?, ?)
                """,
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )

        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

In [2]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import (
    ConsoleSpanExporter,
    SimpleSpanProcessor,
)

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)

trace.set_tracer_provider(provider)
tracer = trace.get_tracer("llm-zoomcamp")

In [3]:
from starter import rag
from rag_helper import RAGBase
class RAGTraced(RAGBase):

    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search"):
            return super().search(query, num_results)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm"):
            response = super().llm(prompt)

            usage = response.usage

            span = trace.get_current_span()
            span.set_attribute("input_tokens", usage.prompt_tokens)
            span.set_attribute("output_tokens", usage.completion_tokens)

            return response

    def rag(self, query):
        with tracer.start_as_current_span("rag"):
            return super().rag(query)

In [4]:
from starter import index, client
rag = RAGTraced(
    index=index,
    llm_client=client,
)

In [10]:
from openai import OpenAI
client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

print(client.models.list())

SyncPage[Model](data=[Model(id='models/gemini-2.5-flash', created=None, object='model', owned_by='google', display_name='Gemini 2.5 Flash'), Model(id='models/gemini-2.5-pro', created=None, object='model', owned_by='google', display_name='Gemini 2.5 Pro'), Model(id='models/gemini-2.0-flash', created=None, object='model', owned_by='google', display_name='Gemini 2.0 Flash'), Model(id='models/gemini-2.0-flash-001', created=None, object='model', owned_by='google', display_name='Gemini 2.0 Flash 001'), Model(id='models/gemini-2.0-flash-lite-001', created=None, object='model', owned_by='google', display_name='Gemini 2.0 Flash-Lite 001'), Model(id='models/gemini-2.0-flash-lite', created=None, object='model', owned_by='google', display_name='Gemini 2.0 Flash-Lite'), Model(id='models/gemini-2.5-flash-preview-tts', created=None, object='model', owned_by='google', display_name='Gemini 2.5 Flash Preview TTS'), Model(id='models/gemini-2.5-pro-preview-tts', created=None, object='model', owned_by='goo

In [7]:
print(rag.model)

models/gemini-3.5-flash


In [12]:
answer = rag.rag(
    "How does the agentic loop keep calling the model until it stops?"
)
print(answer)

Based on the provided context, the agentic loop keeps calling the model using a `while True` loop combined with a tracking flag (such as `has_function_calls`). 

Here is how the mechanism works:

1. **Check for function calls:** In each iteration of the loop, the code sends the message history to the model. It then iterates through the model's output to check if the model has requested any function calls.
2. **Execute tools:** If a function call is requested, the code executes the tool, appends the output back to the conversation history, and sets a flag (e.g., `has_function_calls = True`).
3. **Exit condition:** If the model returns a response that contains no function calls (meaning `has_function_calls` remains `False`), the loop breaks and stops. 

In short, the code continues to make round-trip API calls to the model as long as the model keeps requesting tools, and it stops as soon as the model returns a final answer without any further tool/function calls.
